# **Snorkel Modern Colab — Track 2**
Versión moderna recomendada para Google Colab. Track 2 es más técnico, no “mejor” que Track 1: solo profundiza más. El notebook legado `06_Snorkel.ipynb` queda solo como referencia histórica y comparación.


In [ ]:
!pip install -q snorkel pandas scikit-learn matplotlib

# **Objetivo y contexto**

- **Objetivo**: construir una etiqueta útil a partir de reglas simples (weak supervision) para comenzar un flujo de clasificación con Snorkel.
- **Qué aprenderás**: cómo escribir labeling functions, revisar cobertura/conflictos y entrenar un `LabelModel` a partir de votos imperfectos.
- **Resultado esperado**: un conjunto pequeño de etiquetas automáticas y una evaluación inicial sobre un ejemplo de spam/ham.
- **Por qué Snorkel va después de preparación y augmentación**: primero conviene limpiar, entender y ampliar el texto; después, Snorkel ayuda a convertir ese material en señales de entrenamiento más escalables.
- **Cuándo ayuda**: cuando hay poca anotación manual, reglas de dominio confiables o mucho texto sin etiqueta.
- **Cuándo puede perjudicar**: cuando las reglas son pobres, sesgadas, demasiado rígidas o cubren mal el problema.
- **Ruta recomendada**: use este notebook moderno para publicación. `06_Snorkel.ipynb` queda solo como referencia histórica y comparación.

**Notas de Colab y dependencias**

- La instalación puede tardar varios minutos en Colab.
- Snorkel y sus dependencias pueden ser sensibles a versiones del entorno preinstalado.
- Si aparece un error de compatibilidad, reinicie el runtime y ejecute de nuevo la celda de instalación.
- Si después de reiniciar sigue fallando por resolución de dependencias o drift de versiones, fije temporalmente Snorkel, pandas y scikit-learn a las versiones indicadas en la guía del repositorio y vuelva a ejecutar la instalación.
- Mantenga este notebook sin salidas ni contadores de ejecución para publicar una versión limpia.



In [ ]:
import pandas as pd

from snorkel.labeling import labeling_function
from snorkel.labeling import PandasLFApplier
from snorkel.labeling import LFAnalysis
from snorkel.labeling.model import LabelModel

from sklearn.metrics import classification_report


# **Dataset de ejemplo**
Usamos un conjunto pequeño de mensajes spam/ham para que el flujo sea fácil de seguir en Colab.


In [ ]:
data = [
    ("Check out my channel and subscribe now", "spam"),
    ("This video was really helpful, thank you", "ham"),
    ("Click this link to win a free prize", "spam"),
    ("I learned a lot from this tutorial", "ham"),
    ("Visit my profile for free followers", "spam"),
    ("Great explanation of the topic", "ham"),
    ("Earn money fast with this secret method", "spam"),
    ("Can you explain the second step again?", "ham"),
    ("Free gift card available here", "spam"),
    ("This lesson helped me understand NLP", "ham"),
]

df = pd.DataFrame(data, columns=["text", "label"])
df["label_id"] = df["label"].map({"ham": 0, "spam": 1})

display(df)

# **Etiquetas**


In [ ]:
ABSTAIN = -1
HAM = 0
SPAM = 1

# **Labeling functions**

Las labeling functions son reglas heurísticas. Sirven para arrancar rápido, pero deben diseñarse con cuidado porque codifican supuestos del dominio. Si se definen mal, pueden introducir sesgos sistemáticos, ruido coherente o una cobertura demasiado parcial.


In [ ]:
@labeling_function()
def lf_contains_free(x):
    return SPAM if "free" in x.text.lower() else ABSTAIN

@labeling_function()
def lf_contains_subscribe(x):
    return SPAM if "subscribe" in x.text.lower() else ABSTAIN

@labeling_function()
def lf_contains_click_or_link(x):
    text = x.text.lower()
    return SPAM if ("click" in text or "link" in text) else ABSTAIN

@labeling_function()
def lf_contains_thank_you(x):
    text = x.text.lower()
    return HAM if ("thank you" in text or "helpful" in text) else ABSTAIN

@labeling_function()
def lf_learning_words(x):
    text = x.text.lower()
    learning_terms = ["learned", "tutorial", "lesson", "explain", "understand"]
    return HAM if any(term in text for term in learning_terms) else ABSTAIN

# **Aplicar las funciones**


In [ ]:
lfs = [
    lf_contains_free,
    lf_contains_subscribe,
    lf_contains_click_or_link,
    lf_contains_thank_you,
    lf_learning_words,
]

applier = PandasLFApplier(lfs=lfs)
L_train = applier.apply(df=df)

L_train

# **Analizar cobertura y conflictos**


In [ ]:
analysis = LFAnalysis(L=L_train, lfs=lfs).lf_summary(Y=df["label_id"].values)
display(analysis)

**Cómo leer la tabla**

- **coverage**: cuántos ejemplos etiqueta una función.
- **overlap**: cuántas funciones etiquetan el mismo ejemplo.
- **conflict**: cuándo funciones distintas votan diferente.

Esto ayuda a detectar reglas débiles, redundantes o contradictorias antes de entrenar el modelo de etiquetas.


# **Entrenar LabelModel**


In [ ]:
label_model = LabelModel(cardinality=2, verbose=True)
label_model.fit(L_train=L_train, n_epochs=500, log_freq=100, seed=42)

probs_train = label_model.predict_proba(L=L_train)
preds_train = probs_train.argmax(axis=1)

df["snorkel_label"] = preds_train
df["snorkel_label_name"] = df["snorkel_label"].map({0: "ham", 1: "spam"})

display(df[["text", "label", "snorkel_label_name"]])

# **Evaluar**


In [ ]:
print(classification_report(
    df["label_id"],
    df["snorkel_label"],
    target_names=["ham", "spam"]
))

## **Explicación para personas no técnicas**

La **supervisión débil** combina señales imperfectas —reglas, patrones y heurísticas— para construir una señal de entrenamiento útil. No reemplaza el criterio humano: lo organiza para trabajar con más escala.


#### **Qué hace este notebook**

1. Parte de ejemplos cortos de spam y mensajes normales.
2. Define reglas simples que actúan como votos.
3. Revisa cuánta información aporta cada regla.
4. Combina esos votos en una etiqueta final más estable.
5. Muestra una evaluación inicial para entender el resultado.


#### **Qué aportan las reglas**

Las reglas permiten usar conocimiento práctico del dominio sin etiquetar todo a mano. Por ejemplo, una palabra como “free” puede ser una pista fuerte de spam, mientras que “tutorial” o “thank you” suelen apuntar a mensajes normales.


#### **Cuándo ayuda**

- Cuando hay mucho texto sin etiquetar.
- Cuando existen patrones repetitivos y observables.
- Cuando se necesita una primera versión rápida del dataset.

#### **Cuándo puede perjudicar**

- Cuando las reglas reflejan supuestos incompletos.
- Cuando una regla confunde una señal frecuente con la clase completa.
- Cuando el conjunto de reglas cubre solo una parte del problema y deja fuera casos importantes.

Una mala selección de reglas puede introducir sesgo o ruido de forma consistente.


#### **Cómo interpretar el resultado**

El modelo no “adivina” la verdad absoluta. Aprende a pesar los votos de las reglas y produce una etiqueta probable. Eso sirve como base para seguir refinando el dataset, revisar casos dudosos y mejorar las reglas.


#### **Inspiración y referencias**

- **Practical Natural Language Processing** (O’Reilly)
- https://www.practicalnlp.ai/
- https://github.com/practical-nlp/practical-nlp-code/blob/master/Ch2/

Este material está adaptado y curado con fines de aprendizaje. No implica afiliación oficial con esas fuentes.


#### **Idea clave final**

La supervisión débil combina reglas imperfectas, pero útiles, para crear una señal de entrenamiento más escalable que la anotación manual completa.


#### **Resumen del flujo**

- Preparar un ejemplo pequeño y entendible.
- Diseñar reglas con criterio.
- Medir cobertura y conflictos.
- Entrenar el `LabelModel`.
- Revisar resultados y ajustar.


## **Cierre**

Este notebook muestra una forma práctica de empezar con Snorkel en Colab sin depender de anotación manual total. La meta es acelerar el arranque, no reemplazar el análisis cuidadoso del dominio.
